In [1]:
import requests
import pandas as pd
import numpy as np

from io import StringIO
url = r'https://bit.ly/wine_csv_data'
try:
    r = requests.get(url, timeout=10)
    r.raise_for_status()
    wine = pd.read_csv(StringIO(r.text), low_memory=False)
    
    if wine is not None:
        print(wine.head(10), '\n')
        print(wine.columns, '\n')
        print(wine.info(), '\n')
        print(wine.describe(), '\n')
except Exception as e:
    print(e)
    

   alcohol  sugar    pH  class
0      9.4    1.9  3.51    0.0
1      9.8    2.6  3.20    0.0
2      9.8    2.3  3.26    0.0
3      9.8    1.9  3.16    0.0
4      9.4    1.9  3.51    0.0
5      9.4    1.8  3.51    0.0
6      9.4    1.6  3.30    0.0
7     10.0    1.2  3.39    0.0
8      9.5    2.0  3.36    0.0
9     10.5    6.1  3.35    0.0 

Index(['alcohol', 'sugar', 'pH', 'class'], dtype='object') 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6497 entries, 0 to 6496
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   alcohol  6497 non-null   float64
 1   sugar    6497 non-null   float64
 2   pH       6497 non-null   float64
 3   class    6497 non-null   float64
dtypes: float64(4)
memory usage: 203.2 KB
None 

           alcohol        sugar           pH        class
count  6497.000000  6497.000000  6497.000000  6497.000000
mean     10.491801     5.443235     3.218501     0.753886
std       1.192712     4.757804     0.1

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, cross_validate
from scipy.stats import randint, uniform
from sklearn.ensemble import RandomForestClassifier

wine_i = wine[['alcohol', 'sugar', 'pH']]
wine_t = wine['class']

train_i, test_i, train_t, test_t = train_test_split(wine_i, wine_t, test_size=0.2, stratify=wine_t)
rf = RandomForestClassifier(random_state=42)
splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_validate(rf, train_i, train_t, cv=splitter, return_train_score=True)
print(np.mean(scores['train_score']))
print(np.mean(scores['test_score']))

params = {'n_estimators': randint(200, 800),
    'max_depth': randint(3, 25),
    'min_samples_split': randint(2, 30),
    'min_samples_leaf': randint(1, 20),
    'max_features': ['sqrt', 'log2', None],
    'ccp_alpha': uniform(0.0, 0.01)}

rs = RandomizedSearchCV(RandomForestClassifier(random_state=42), params, n_iter=100, cv=splitter, random_state=42)
rs.fit(train_i, train_t)
rf2 = rs.best_estimator_
print(rs.best_params_)


0.9972580774120345
0.8903239061227511
{'ccp_alpha': np.float64(0.00043400783298172763), 'max_depth': 23, 'max_features': None, 'min_samples_leaf': 5, 'min_samples_split': 5, 'n_estimators': 426}


KeyError: 'test_score'

In [12]:
rf = RandomForestClassifier(random_state=42)
scores = cross_validate(rf, train_i, train_t, cv=splitter, return_train_score=True)
print(np.mean(scores['train_score']), np.mean(scores['test_score']))

params = {'n_estimators': randint(200, 500),
    'max_depth': randint(3, 25),
    'min_samples_split': randint(2, 30),
    'min_samples_leaf': randint(1, 20),
    }
from sklearn.ensemble import ExtraTreesClassifier
rs = RandomizedSearchCV(ExtraTreesClassifier(random_state=42),
                        params,
                        cv=splitter,
                        random_state=42,
                        n_iter=100)
rs.fit(train_i, train_t)                        
print(rs.best_params_)
print(np.max(rs.cv_results_['mean_test_score']))
et = rs.best_estimator_
et.fit(train_i, train_t)
print(et.score(train_i, train_t), et.score(test_i, test_t))


0.9970656772196342 0.8858963870585622


KeyboardInterrupt: 

In [19]:
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import SGDClassifier
# 0.9247162623636042 0.8699267046716518 / 0.9253898365998439 0.8681959354408825
gb = GradientBoostingClassifier(n_estimators=300, learning_rate=0.2, loss='log_loss', random_state=42, subsample=0.7) 
scores = cross_validate(gb, train_i, train_t, cv=splitter, return_train_score=True )
print(np.mean(scores['train_score']), np.mean(scores['test_score']))


hgb = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.2, random_state=42, loss='log_loss')
scores = cross_validate(hgb, train_i, train_t, cv=splitter, return_train_score=True)
print(np.mean(scores['train_score']), np.mean(scores['test_score']))

from sklearn.inspection import permutation_importance
hgb.fit(train_i, train_t)
gb.fit(train_i, train_t)
result = permutation_importance(hgb, train_i, train_t, n_repeats=10, random_state=42)
print(hgb.feature_names_in_)
print(result)
print(gb.feature_importances_)


0.9253898365998439 0.8681959354408825
0.9934097148674969 0.8776249352187755
['alcohol' 'sugar' 'pH']
{'importances_mean': array([0.15891861, 0.29636329, 0.16136232]), 'importances_std': array([0.00418965, 0.00320918, 0.00341206]), 'importances': array([[0.16201655, 0.15816817, 0.16086204, 0.15143352, 0.16066962,
        0.15682124, 0.15605157, 0.15970752, 0.16778911, 0.15566673],
       [0.29786415, 0.29709448, 0.29286127, 0.29151434, 0.30286704,
        0.29767173, 0.29613238, 0.29228401, 0.2972869 , 0.29805657],
       [0.15970752, 0.16471041, 0.1625938 , 0.16297864, 0.16374832,
        0.16201655, 0.16124687, 0.15759092, 0.16548008, 0.15355013]])}
[0.20012341 0.61422948 0.18564711]
